In [3]:
import torch
import torch.nn.functional as F
from einops import rearrange

torch.set_printoptions(precision=4, sci_mode=False)
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32

In [4]:
device

'cuda'

In [15]:
def segsum(x: torch.Tensor) -> torch.Tensor:
    """Naive segment sum calc. exp(segsum(A)) gives lower-triangular exp(cumsum diffs)."""
    T = x.size(-1)
    x_cumsum = torch.cumsum(x, dim=-1)
    print(x_cumsum.shape)
    print(x_cumsum)
    x_segsum = x_cumsum[..., :, None] - x_cumsum[..., None, :]
    print(x_segsum.shape)
    print(x_segsum)
    mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=0)
    print(mask)
    x_segsum = x_segsum.masked_fill(~mask, -torch.inf)
    return x_segsum

In [6]:
def ssd(X, A, B, C, block_len=64, initial_states=None, debug=False):
    """
    X: (b, L, h, p)      p=d_head
    A: (b, L, h)         (log-decay per step; scalar SSM per head)
    B: (b, L, h, n)      n=d_state
    C: (b, L, h, n)
    Returns:
      Y: (b, L, h, p)
      final_state: (b, h, p, n)   (boundary after last chunk)
    """
    assert X.dtype == A.dtype == B.dtype == C.dtype
    assert X.shape[1] % block_len == 0

    # chunk into (b, c, l, ...)
    Xc, Ac, Bc, Cc = [rearrange(x, "b (c l) ... -> b c l ...", l=block_len) for x in (X, A, B, C)]
    # move head to match the original snippet usage
    Ac_h = rearrange(Ac, "b c l h -> b h c l")
    A_cumsum = torch.cumsum(Ac_h, dim=-1)  # (b,h,c,l)

    if debug:
        print("Xc:", Xc.shape, "Ac_h:", Ac_h.shape, "Bc:", Bc.shape, "Cc:", Cc.shape)
        print("A_cumsum:", A_cumsum.shape)

    # 1) intra-chunk (diagonal blocks)
    L = torch.exp(segsum(Ac_h))  # (b,h,c,l,s)
    Y_diag = torch.einsum("bclhn,bcshn,bhcls,bcshp->bclhp", Cc, Bc, L, Xc)

    # 2) per-chunk boundary state contributions (right factor)
    decay_states = torch.exp((A_cumsum[:, :, :, -1:] - A_cumsum))  # (b,h,c,l)
    states = torch.einsum("bclhn,bhcl,bclhp->bchpn", Bc, decay_states, Xc)  # (b,c,h,p,n)

    # 3) inter-chunk scan across boundaries (middle factor)
    if initial_states is None:
        initial_states = torch.zeros_like(states[:, :1])  # (b,1,h,p,n)
    states_bound = torch.cat([initial_states, states], dim=1)      # (b,c+1,h,p,n)

    A_end = A_cumsum[:, :, :, -1]                     # (b,h,c)
    A_end_pad = F.pad(A_end, (1, 0))                  # (b,h,c+1)
    decay_chunk = torch.exp(segsum(A_end_pad))         # (b,h,z,c) where z,c are boundaries
    new_states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states_bound)  # (b,z,h,p,n)

    states_in = new_states[:, :-1]      # boundary at start of each chunk: (b,c,h,p,n)
    final_state = new_states[:, -1]     # after last chunk boundary: (b,h,p,n)

    # 4) boundary-state -> output inside each chunk (left factor)
    state_decay_out = torch.exp(A_cumsum)  # (b,h,c,l)
    Y_off = torch.einsum("bclhn,bchpn,bhcl->bclhp", Cc, states_in, state_decay_out)

    Y = rearrange(Y_diag + Y_off, "b c l h p -> b (c l) h p")
    return Y, final_state

In [7]:
# ---- Run a test ----
torch.manual_seed(0)

b = 2
L = 128          # must be multiple of block_len
h = 3
p = 4
n = 5
block_len = 64

X = torch.randn(b, L, h, p, device=device, dtype=dtype)
# keep A negative-ish for stable decays (optional)
A = -0.1 * torch.rand(b, L, h, device=device, dtype=dtype)
B = torch.randn(b, L, h, n, device=device, dtype=dtype)
C = torch.randn(b, L, h, n, device=device, dtype=dtype)

Y, final_state = ssd(X, A, B, C, block_len=block_len, debug=True)
print("Y:", Y.shape, "final_state:", final_state.shape)

# Sanity-check vs naive on a SMALLER L (optional)
L_small = 32
block_len_small = 16
X_s = X[:, :L_small].contiguous()
A_s = A[:, :L_small].contiguous()
B_s = B[:, :L_small].contiguous()
C_s = C[:, :L_small].contiguous()

Y_fast, _ = ssd(X_s, A_s, B_s, C_s, block_len=block_len_small)

Xc: torch.Size([2, 2, 64, 3, 4]) Ac_h: torch.Size([2, 3, 2, 64]) Bc: torch.Size([2, 2, 64, 3, 5]) Cc: torch.Size([2, 2, 64, 3, 5])
A_cumsum: torch.Size([2, 3, 2, 64])
Y: torch.Size([2, 128, 3, 4]) final_state: torch.Size([2, 3, 4, 5])


In [8]:
Xc, Ac, Bc, Cc = [rearrange(x, "b (c l) ... -> b c l ...", l=block_len) for x in (X, A, B, C)]
# move head to match the original snippet usage
Ac_h = rearrange(Ac, "b c l h -> b h c l")
A_cumsum = torch.cumsum(Ac_h, dim=-1)  # (b,h,c,l)

In [16]:
# 1) intra-chunk (diagonal blocks)
L = torch.exp(segsum(Ac_h))  # (b,h,c,l,s)

torch.Size([2, 3, 2, 64])
tensor([[[[-0.0972, -0.1302, -0.2066, -0.2426, -0.3035, -0.3934, -0.4631,
           -0.5458, -0.5787, -0.6537, -0.6710, -0.7545, -0.8314, -0.8466,
           -0.9416, -0.9455, -0.9465, -1.0058, -1.0641, -1.1422, -1.2051,
           -1.2970, -1.3298, -1.3552, -1.4114, -1.4686, -1.5622, -1.6558,
           -1.6857, -1.7205, -1.7928, -1.8861, -1.9326, -2.0276, -2.1030,
           -2.2004, -2.2386, -2.3191, -2.3278, -2.3380, -2.4097, -2.4157,
           -2.4500, -2.5003, -2.5307, -2.6082, -2.6796, -2.7120, -2.7660,
           -2.8026, -2.8420, -2.9223, -2.9404, -3.0240, -3.0405, -3.1293,
           -3.2039, -3.2454, -3.2960, -3.3732, -3.4700, -3.4979, -3.5517,
           -3.6298],
          [-0.0248, -0.0730, -0.0983, -0.1207, -0.2085, -0.2225, -0.2644,
           -0.2828, -0.3289, -0.3337, -0.3919, -0.4790, -0.5451, -0.5749,
           -0.6461, -0.6849, -0.7654, -0.7761, -0.7944, -0.8153, -0.8624,
           -0.9600, -1.0401, -1.1267, -1.1486, -1.2038, -1.2148, 

In [17]:
Xc.shape

torch.Size([2, 2, 64, 3, 4])

In [19]:
Y_diag = torch.einsum("bclhn,bcshn,bhcls,bcshp->bclhp", Cc, Bc, L, Xc)
Y_diag.shape

torch.Size([2, 2, 64, 3, 4])

In [20]:
# 2) per-chunk boundary state contributions (right factor)
decay_states = torch.exp((A_cumsum[:, :, :, -1:] - A_cumsum))  # (b,h,c,l)
states = torch.einsum("bclhn,bhcl,bclhp->bchpn", Bc, decay_states, Xc)  # (b,c,h,p,n)

In [27]:
# 3) inter-chunk scan across boundaries (middle factor)
initial_states = None
if initial_states is None:
    initial_states = torch.zeros_like(states[:, :1])  # (b,1,h,p,n)
states_bound = torch.cat([initial_states, states], dim=1)      # (b,c+1,h,p,n)

A_end = A_cumsum[:, :, :, -1]                     # (b,h,c)
A_end_pad = F.pad(A_end, (1, 0))                  # (b,h,c+1)
decay_chunk = torch.exp(segsum(A_end_pad))         # (b,h,z,c) where z,c are boundaries
new_states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states_bound)  # (b,z,h,p,n)

states_in = new_states[:, :-1]      # boundary at start of each chunk: (b,c,h,p,n)
final_state = new_states[:, -1]     # after last chunk boundary: (b,h,p,n)

torch.Size([2, 3, 3])
tensor([[[ 0.0000, -3.6298, -6.6796],
         [ 0.0000, -3.1015, -6.2138],
         [ 0.0000, -3.1612, -6.6689]],

        [[ 0.0000, -2.7571, -5.8413],
         [ 0.0000, -3.2692, -6.5599],
         [ 0.0000, -3.2694, -6.4489]]], device='cuda:0')
torch.Size([2, 3, 3, 3])
tensor([[[[ 0.0000,  3.6298,  6.6796],
          [-3.6298,  0.0000,  3.0498],
          [-6.6796, -3.0498,  0.0000]],

         [[ 0.0000,  3.1015,  6.2138],
          [-3.1015,  0.0000,  3.1123],
          [-6.2138, -3.1123,  0.0000]],

         [[ 0.0000,  3.1612,  6.6689],
          [-3.1612,  0.0000,  3.5077],
          [-6.6689, -3.5077,  0.0000]]],


        [[[ 0.0000,  2.7571,  5.8413],
          [-2.7571,  0.0000,  3.0842],
          [-5.8413, -3.0842,  0.0000]],

         [[ 0.0000,  3.2692,  6.5599],
          [-3.2692,  0.0000,  3.2907],
          [-6.5599, -3.2907,  0.0000]],

         [[ 0.0000,  3.2694,  6.4489],
          [-3.2694,  0.0000,  3.1795],
          [-6.4489, -3.1795, 

In [34]:
decay_chunk

tensor([[[[1.0000, 0.0000, 0.0000],
          [0.0265, 1.0000, 0.0000],
          [0.0013, 0.0474, 1.0000]],

         [[1.0000, 0.0000, 0.0000],
          [0.0450, 1.0000, 0.0000],
          [0.0020, 0.0445, 1.0000]],

         [[1.0000, 0.0000, 0.0000],
          [0.0424, 1.0000, 0.0000],
          [0.0013, 0.0300, 1.0000]]],


        [[[1.0000, 0.0000, 0.0000],
          [0.0635, 1.0000, 0.0000],
          [0.0029, 0.0458, 1.0000]],

         [[1.0000, 0.0000, 0.0000],
          [0.0380, 1.0000, 0.0000],
          [0.0014, 0.0372, 1.0000]],

         [[1.0000, 0.0000, 0.0000],
          [0.0380, 1.0000, 0.0000],
          [0.0016, 0.0416, 1.0000]]]], device='cuda:0')

In [31]:
decay_chunk.shape

torch.Size([2, 3, 3, 3])

In [35]:



# 4) boundary-state -> output inside each chunk (left factor)
state_decay_out = torch.exp(A_cumsum)  # (b,h,c,l)
Y_off = torch.einsum("bclhn,bchpn,bhcl->bclhp", Cc, states_in, state_decay_out)

Y = rearrange(Y_diag + Y_off, "b c l h p -> b (c l) h p")

In [36]:
state_decay_out

tensor([[[[0.9074, 0.8779, 0.8133, 0.7846, 0.7382, 0.6748, 0.6293, 0.5794,
           0.5606, 0.5201, 0.5112, 0.4703, 0.4355, 0.4289, 0.3900, 0.3885,
           0.3881, 0.3658, 0.3450, 0.3191, 0.2997, 0.2733, 0.2645, 0.2579,
           0.2438, 0.2302, 0.2097, 0.1909, 0.1853, 0.1790, 0.1665, 0.1517,
           0.1448, 0.1317, 0.1221, 0.1108, 0.1066, 0.0984, 0.0975, 0.0965,
           0.0898, 0.0893, 0.0863, 0.0821, 0.0796, 0.0737, 0.0686, 0.0664,
           0.0629, 0.0607, 0.0583, 0.0538, 0.0528, 0.0486, 0.0478, 0.0437,
           0.0406, 0.0390, 0.0370, 0.0343, 0.0311, 0.0303, 0.0287, 0.0265],
          [0.9755, 0.9296, 0.9064, 0.8863, 0.8118, 0.8005, 0.7677, 0.7537,
           0.7197, 0.7162, 0.6758, 0.6194, 0.5798, 0.5628, 0.5241, 0.5041,
           0.4652, 0.4602, 0.4519, 0.4425, 0.4221, 0.3829, 0.3534, 0.3241,
           0.3171, 0.3001, 0.2968, 0.2961, 0.2830, 0.2687, 0.2651, 0.2651,
           0.2445, 0.2361, 0.2155, 0.2134, 0.2028, 0.1894, 0.1766, 0.1602,
           0.1527, 0.139